# DocLite: Multimodal Distillation (Image-Enhanced Teacher)

This notebook demonstrates the **key insight** of DocLite:

- **Teacher (LayoutLMv3)** sees **text + layout + image** (full multimodal)
- **Student (LiLT)** sees only **text + layout** (no image needed)
- Through distillation, the student learns image-informed representations **without ever seeing images**

This means at inference time, the student is both:
1. **Smaller** (fewer parameters than LayoutLMv3)
2. **Simpler** (no image processing pipeline needed)

...while still benefiting from the teacher's visual understanding.

We compare 3 setups on FUNSD:

| Model | Inputs | Training |
|-------|--------|----------|
| LayoutLMv3 (teacher) | text + layout + **image** | Supervised |
| LiLT baseline | text + layout | Supervised |
| LiLT + DocLite | text + layout | Distilled from multimodal teacher |

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LayoutLMv3Config, LayoutLMv3ForTokenClassification

from doclite.configs.core import ENV, WEIGHTS
from doclite.data.document_dataset import DocumentDataset
from doclite.data.collate import collate_document_batch
from doclite.eval.evaluate import evaluate_student
from doclite.models.teacher import TeacherModel
from doclite.models.student import StudentModel
from doclite.distill.distill_loss import compute_distill_loss
from build_funsd_examples import load_funsd_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load FUNSD with Images

In [ ]:
FUNSD_ROOT = ENV.DATA / "funsd"
TRAIN_ANN_DIR = FUNSD_ROOT / "training_data" / "annotations"
TRAIN_IMG_DIR = FUNSD_ROOT / "training_data" / "images"
TEST_ANN_DIR = FUNSD_ROOT / "testing_data" / "annotations"
TEST_IMG_DIR = FUNSD_ROOT / "testing_data" / "images"

tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv3-base")

# Load WITH images (pass image_dir)
print("Loading training data with images...")
train_examples = load_funsd_split(TRAIN_ANN_DIR, tokenizer, image_dir=TRAIN_IMG_DIR)
print("Loading test data with images...")
test_examples = load_funsd_split(TEST_ANN_DIR, tokenizer, image_dir=TEST_IMG_DIR)

print(f"Training documents: {len(train_examples)}")
print(f"Testing documents:  {len(test_examples)}")
print(f"Has pixel_values: {'pixel_values' in train_examples[0]}")
if 'pixel_values' in train_examples[0]:
    import torch as t
    pv = t.tensor(train_examples[0]['pixel_values'])
    print(f"pixel_values shape: {pv.shape}  (channels, height, width)")

In [ ]:
BATCH_SIZE = 2
NUM_LABELS = 4
NUM_EPOCHS = 3
LR = 1e-5

train_dataset = DocumentDataset(train_examples)
test_dataset = DocumentDataset(test_examples)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

## 2. LayoutLMv3 Baseline with Images (Teacher Upper Bound)

In [ ]:
config = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=NUM_LABELS,
    output_hidden_states=True,
    output_attentions=True,
)

teacher_model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config
).to(device)

optimizer = torch.optim.AdamW(teacher_model.parameters(), lr=LR, weight_decay=0.01)

class LayoutLMv3EvalWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, bbox, **kwargs):
        # Pass pixel_values if available
        fwd_kwargs = dict(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox)
        if 'pixel_values' in kwargs:
            fwd_kwargs['pixel_values'] = kwargs['pixel_values']
        out = self.model(**fwd_kwargs)
        return {"logits": out.logits}

eval_wrapper = LayoutLMv3EvalWrapper(teacher_model).to(device)
print(f"LayoutLMv3 parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")

In [ ]:
teacher_results = []

for epoch in range(NUM_EPOCHS):
    teacher_model.train()
    total_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        outputs = teacher_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            pixel_values=batch.get("pixel_values"),
            labels=batch["labels"],
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 25 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    metrics = evaluate_student(eval_wrapper, test_loader, device=device)
    teacher_results.append(metrics)

    print(f"Epoch {epoch+1} | train_loss={avg_loss:.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\nLayoutLMv3+Image Best F1: {max(r['token_f1'] for r in teacher_results):.4f}")

## 3. Multimodal Distillation

Teacher receives `pixel_values` (sees the document image).  
Student receives only `input_ids`, `attention_mask`, `bbox` (text + layout only).  
The student learns from the teacher's image-informed hidden states, attentions, and logits.

In [ ]:
# Frozen teacher with image support
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS).to(device)
for param in teacher.parameters():
    param.requires_grad = False

# Student (text + layout only)
student_distill = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS).to(device)
optimizer_sd = torch.optim.AdamW(student_distill.parameters(), lr=LR, weight_decay=0.01)

print(f"Teacher params: {sum(p.numel() for p in teacher.parameters()):,}")
print(f"Student params: {sum(p.numel() for p in student_distill.parameters()):,}")
print(f"Compression ratio: {sum(p.numel() for p in teacher.parameters()) / sum(p.numel() for p in student_distill.parameters()):.1f}x")

In [ ]:
distill_results = []
best_f1 = -1.0

for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student_distill.train()

    total_loss = 0.0
    total_distill = 0.0
    total_task = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer_sd.zero_grad()

        # Teacher gets FULL multimodal input (text + layout + image)
        teacher_inputs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"],
            "bbox": batch["bbox"],
        }
        if "pixel_values" in batch:
            teacher_inputs["pixel_values"] = batch["pixel_values"]

        teacher_out = teacher(**teacher_inputs)

        # Student gets ONLY text + layout (no image)
        student_hf_out = student_distill.model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            labels=batch["labels"],
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )

        student_out = {
            "logits": student_hf_out.logits,
            "hidden_states": student_hf_out.hidden_states,
            "attentions": student_hf_out.attentions,
        }

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        distill_loss = distill_losses["loss_total"]
        task_loss = student_hf_out.loss

        loss = distill_loss + WEIGHTS.DELTA_TASK * task_loss
        loss.backward()
        optimizer_sd.step()

        total_loss += loss.item()
        total_distill += distill_loss.item()
        total_task += task_loss.item()

        if step % 25 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | total={loss.item():.4f} | distill={distill_loss.item():.4f} | task={task_loss.item():.4f}")

    avg_total = total_loss / len(train_loader)
    metrics = evaluate_student(student_distill, test_loader, device=device)
    distill_results.append(metrics)

    if metrics["token_f1"] > best_f1:
        best_f1 = metrics["token_f1"]

    print(f"Epoch {epoch+1} | train_loss={avg_total:.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

print(f"\nMultimodal DocLite Best F1: {best_f1:.4f}")

## 4. Results Comparison

In [ ]:
print("=" * 70)
print(f"{'Model':<35s} {'Inputs':<15s} {'Acc':>8s} {'F1':>8s}")
print("=" * 70)

teacher_best = max(teacher_results, key=lambda x: x['token_f1'])
distill_best = max(distill_results, key=lambda x: x['token_f1'])

print(f"{'LayoutLMv3 (teacher)':<35s} {'T+L+Image':<15s} {teacher_best['token_acc']:>8.4f} {teacher_best['token_f1']:>8.4f}")
print(f"{'LiLT + DocLite (multimodal)':<35s} {'T+L only':<15s} {distill_best['token_acc']:>8.4f} {distill_best['token_f1']:>8.4f}")
print("=" * 70)
print()
print("Key: T=Text, L=Layout")
print("Note: The student NEVER sees images, yet benefits from the teacher's visual understanding.")